In [36]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [37]:
load_dotenv()

X_train_path = os.getenv("X_TRAIN_CSV_PATH")
y_train_path = os.getenv("Y_TRAIN_TSV_PATH")
X_test_path = os.getenv("X_TEST_CSV_PATH")
y_test_path = os.getenv("Y_TEST_TSV_PATH")
model_path = os.getenv("RF_MODEL_PATH")
selected_features_path = os.getenv("SELECTED_FEATURES_PATH")
hyperparameters_path = os.getenv("HYPERPARAMETERS_PATH")
pipeline_path = os.getenv("PIPELINE_PATH")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"model_path: {model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"hyperparameters_path: {hyperparameters_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")

# Hardcoded variables
VARIANCE_THRESHOLD=0.01  # REMOVE NEAR-CONSTANT GENES
FEATURE_SELECTION_PERCENTILE=20  # KEEP TOP n% OF FEATURES
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/transpose/transposed.csv
y_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/class/subtypes.tsv
X_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/transpose/transposed.csv
y_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/class/subtypes.tsv
model_path: /home/mariya/Data/pallas/training/gex/gex.model.pkl
selected_features_path: /home/mariya/Data/pallas/training/gex/gex.selected_features.pkl
hyperparameters_path: /home/mariya/Data/pallas/training/gex/gex.hyperparameters.pkl
pipeline_path: /home/mariya/Data/pallas/training/gex/gex.pipeline.pkl
random_seed: 42
n_jobs: 10


In [38]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
id,,,,,,,,,,,,,,,,,,,,,
AML011-D,-1.282099,-1.282099,4.226514,4.073582,4.442745,3.728175,-0.981534,4.310436,4.372199,5.847456,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
AML015-D,-0.977991,-1.282099,3.953264,4.731984,4.077643,6.146657,-1.094207,4.288357,4.121187,6.015334,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
AML023-D,-1.083304,-1.282099,4.414604,3.433837,4.254018,9.054867,0.033352,5.052131,3.987931,5.845450,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
AML035-D,-0.935303,-1.282099,4.700613,3.824039,4.594002,5.187068,2.718534,4.710378,4.145544,6.402966,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
AML036-D,-0.260191,-1.282099,4.436169,3.991077,4.181627,6.826557,4.617441,4.194369,4.427726,6.439500,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJPHALL020044,-1.282099,-1.282099,3.605629,4.287436,4.553397,8.745959,3.879038,6.386409,5.045250,5.801866,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
SJPHALL021,-1.282099,-1.282099,3.960824,4.443220,4.189139,7.898405,5.797672,6.938610,5.418718,6.050723,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
SJPHALL022668,-1.282099,-1.282099,5.294222,3.946928,3.191243,5.585420,3.240574,4.522400,4.764555,4.996434,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099


In [39]:
y_train = pd.read_csv(y_train_path, sep="\t", index_col=0)
y_train

,class
sample,
SJBALL013921,"B-ALL, KMT2A-r"
SJBALL041794,"B-ALL, BCR::ABL1"
SJBALL048070,"B-ALL, DUX4-r"
SJBALL013345,"B-ALL, ETV6::RUNX1-like"
SJBALL013369,"B-ALL, BCR::ABL1-like"
...,...
SJHBBBM2GV5,"B-ALL, ETV6::RUNX1"
SJHBBBM55F5,"B-ALL, high hyperdiploid"
SJHBBBN9RFN,"B-ALL, high hyperdiploid"


In [40]:
# align the indices
X_train = X_train.loc[y_train.index]
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJBALL013921,-0.504928,-1.282099,4.462993,2.829355,3.137464,4.307821,-0.397533,5.975342,3.684384,6.442814,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
SJBALL041794,-0.782360,-1.282099,3.010584,4.021878,3.693740,4.428879,5.453373,3.395220,4.765206,7.104321,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
SJBALL048070,-0.847124,-1.282099,4.374241,3.867942,3.791044,7.475785,2.725818,4.193459,4.402564,6.419607,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
SJBALL013345,-1.001948,-1.282099,4.256353,3.352391,3.870006,6.226826,-0.464664,3.856711,3.125386,6.157760,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
SJBALL013369,-0.327413,-1.282099,5.227689,3.049944,3.690913,6.424426,3.138618,4.226766,5.123261,5.922237,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJHBBBM2GV5,-1.235923,-1.282099,3.815016,4.220331,3.716659,5.650445,1.494295,3.236938,4.207741,6.187247,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
SJHBBBM55F5,-0.483354,-1.282099,4.084989,4.257419,4.473667,5.376888,2.362712,4.269596,5.219486,4.609465,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099
SJHBBBN9RFN,0.421300,-1.282099,3.908848,3.590228,4.149405,7.040323,3.368121,5.451227,6.133563,5.971046,...,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099,-1.282099


In [41]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
y_test

,class
sample,
SJBALL033899_D1,"B-ALL, high hyperdiploid"
SJAML030781_D1,"AML, KMT2A-r"
SJCBF025_D,"AML, CBFB::MYH11"
SJBALL030207_D1,"B-ALL, ETV6::RUNX1"
SJTALL030442_D1,T-ALL
...,...
SJHBBBM5PY5,"B-ALL, high hyperdiploid"
SJHBBBM588V,"B-ALL, high hyperdiploid"
SJHBBBM67H6,"B-ALL, high hyperdiploid"


In [42]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJBALL033899_D1,-0.380216,-1.026671,4.708468,4.151959,5.252940,4.027415,4.306655,1.431260,2.982075,6.594644,...,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388
SJAML030781_D1,0.872116,2.544027,4.578276,3.813292,3.670275,6.705403,1.808781,5.109912,4.897542,4.801703,...,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388
SJCBF025_D,-1.117743,-1.253388,4.116243,4.350195,4.146378,6.815896,2.725745,4.533234,4.738120,6.396030,...,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388
SJBALL030207_D1,-0.889019,-1.253388,4.293268,4.328442,4.884145,4.886754,3.157192,1.615156,3.131022,6.518226,...,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388
SJTALL030442_D1,0.298616,-1.253388,4.196140,3.724836,5.601207,4.775002,1.633179,3.059045,6.304201,7.335021,...,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJHBBBM5PY5,0.465597,-1.253388,4.803914,1.986555,3.821151,4.210445,3.902285,2.996705,5.368200,7.100883,...,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388
SJHBBBM588V,-0.815576,-1.253388,3.563414,4.184634,4.543960,3.051011,3.746603,4.671647,5.073919,6.064630,...,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388
SJHBBBM67H6,0.338338,-1.253388,4.258116,1.213634,3.511781,4.285381,4.051735,2.304520,3.284695,6.600976,...,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388,-1.253388


In [43]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(3)

In [44]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_test = y_test.iloc[:, 0].values
y_test

array(['B-ALL, high hyperdiploid', 'AML, KMT2A-r', 'AML, CBFB::MYH11',
       'B-ALL, ETV6::RUNX1', 'T-ALL', 'B-ALL, ETV6::RUNX1',
       'AML, CBFB::MYH11', 'AML, CBFB::MYH11', 'T-ALL',
       'B-ALL, ETV6::RUNX1', 'T-ALL', 'B-ALL, ETV6::RUNX1',
       'AML, KMT2A-r', 'T-ALL', 'B-ALL, BCR::ABL1',
       'B-ALL, high hyperdiploid', 'AML, CBFB::MYH11', 'AML, DEK::NUP214',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
       'AML, RUNX1::RUNX1T1', 'B-ALL, KMT2A-r', 'AML, RUNX1::RUNX1T1',
       'AML, RBM15::MKL1', 'T-ALL', 'AML, RUNX1::RUNX1T1',
       'B-ALL, KMT2A-r', 'B-ALL, DUX4-r', 'B-ALL, ETV6::RUNX1',
       'AML, RUNX1::RUNX1T1', 'AML, RUNX1::RUNX1T1', 'AML, KMT2A-r',
       'AML, NUP98-r', 'B-ALL, ZNF384-r', 'B-ALL, high hyperdiploid',
       'B-ALL, TCF3::PBX1', 'AML, NPM1alt', 'AML, KMT2A-r',
       'AML, NUP98-r', 'AML, CBFA2T3::GLIS2', 'B-ALL, BCR::ABL1',
       'AML, KMT2A-r', 'B-ALL, DUX4-r', 'T-ALL', 'B-ALL, ETV6::RUNX1',
       'AML, RUNX1::RUNX1T1', 'B-ALL, h

In [45]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=VARIANCE_THRESHOLD)), # remove near-constant genes
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': [None, 'balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=-n_jobs,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparameters_path)

# Save features
# First get features that passed the variance threshold
variance_mask = best_pipeline.named_steps['variance_filter'].get_support()
remaining_features = X_train.columns[variance_mask]

# Then get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = remaining_features[selected_mask]

joblib.dump(selected_features, selected_features_path)

# Save the whole pipeline
joblib.dump(best_pipeline, pipeline_path)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best hyperparameters: {'rf__n_estimators': 200, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': 50, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9058911351227458


['/home/mariya/Data/pallas/training/gex/gex.pipeline.pkl']

In [46]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Read in "best_hyperparameters.pkl"
with open(hyperparameters_path, "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=-n_jobs,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, model_path)

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 50, 'class_weight': 'balanced'}


[Parallel(n_jobs=-10)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=-10)]: Done  30 out of  30 | elapsed:  6.4min finished


Cross-validation F1-weighted: 0.910 ± 0.008


['/home/mariya/Data/pallas/training/gex/gex.model.pkl']

In [47]:
# ----------------------
# Evaluate final model on test set
# ----------------------

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))


Number of selected features: 3919
Shape of X_test_selected: (750, 3919)

Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     1.000     1.000         1
     AML, CBFA2T3::GLIS2      0.750     1.000     0.857         6
        AML, CBFB::MYH11      0.969     1.000     0.984        31
           AML, CEBPAalt      0.923     0.923     0.923        13
        AML, DEK::NUP214      0.800     0.667     0.727         6
             AML, ETV6-r      1.000     0.500     0.667         2
           AML, FUS::ERG      1.000     1.000     1.000         1
            AML, KAT6A-r      1.000     1.000     1.000         2
            AML, KMT2A-r      0.909     1.000     0.952        60
            AML, MECOM-r      1.000     1.000     1.000         1
            AML, NPM1alt      0.875     0.955     0.913        22
            AML, NUP98-r      1.000     0.750     0.857        16
        AML,

In [48]:

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
         Gene  Importance
3081    NUTM1    0.004756
1769   MEGF10    0.003917
3619    APELA    0.003496
516     SALL4    0.003298
1715    MEIS1    0.003052
3126     PBX1    0.002571
2110  IGF2BP1    0.002535
2069    AGAP1    0.002338
2824  KBTBD11    0.002326
137   ARHGAP6    0.002291
788     FOLR1    0.002252
1187    KRT17    0.002247
1190     CPA4    0.002222
686    PTPRZ1    0.002169
2719    PTPRM    0.002124
3720    ANXA8    0.002101
743     HOXB6    0.002093
2059     DGKI    0.002054
335   RUNX1T1    0.002026
3633   HOXA10    0.002019


In [49]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 0 out of 3919

Bottom 20 genes by importance (final model):
            Gene    Importance
2401     CCDC182  7.494687e-06
3722      TIMM23  7.366831e-06
3615      ZCCHC3  7.272178e-06
3742         NaN  7.179565e-06
3821        XKR5  6.318988e-06
3735         NaN  5.862572e-06
3790       CWC25  5.741423e-06
3107      RAB11B  5.552119e-06
3644         NaN  5.263078e-06
3006       OR4Q3  4.611024e-06
1520       LHCGR  4.472755e-06
3892     TUBA3FP  3.922101e-06
3640      OR10G4  3.871547e-06
3618         NaN  3.844286e-06
3674  BUB1B-PAK6  3.576078e-06
3914        F8A1  3.149576e-06
3671         NaN  2.558709e-06
3754         NaN  2.267618e-06
1952       PTPRR  9.457498e-08
3696         NaN  1.937598e-20
